# 3-way ブレンド評価 ― v10(LGBM) × v12(TCN) × v13(U-Net)

3モデルの OOF を加重平均し、overall RMSE が下がるか確認(系統: 決定木 / 1D系列 / 2D画像)。

- v10/v12 OOF は parquet 行順・`target = TVT - last_tvt` 空間。
- v13 OOF は `pred_rel`(=予測TVT − center)。`center`/`last_tvt`/`ev_index`(画像npz)で
  **v10空間(TVT − last_tvt)・parquet行順に整列**してから合成する。
- target基準の overall RMSE(=TVT RMSE)。重みは単体グリッド探索。

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

PROJECT = Path(r"C:\Users\kosaka256\Documents\rogii_claude")
PARQUET = PROJECT / "artifacts_v10" / "data" / "train.parquet"
OOF_LGBM = PROJECT / "artifacts_v10" / "oof_fwd_bwd.npy"
OOF_TCN  = PROJECT / "artifacts_v12" / "oof_tcn.npy"
# v13 (パスが2通りあるため自動検出)
def _find(*cands):
    return next((c for c in cands if c.exists()), None)
OOF_UNET = _find(PROJECT/"v13"/"artifacts_v13"/"oof_unet.npz", PROJECT/"artifacts_v13"/"oof_unet.npz")
IMG_DIR  = _find(PROJECT/"v13"/"artifacts_v13"/"images", PROJECT/"artifacts_v13"/"images",
                 PROJECT/"v13"/"artifacts_v13"/"img", PROJECT/"artifacts_v13"/"img")
print("OOF_UNET:", OOF_UNET); print("IMG_DIR:", IMG_DIR)

t = pq.read_table(PARQUET, columns=["id", "target"]).to_pandas()
ids = t["id"].to_numpy(); y = t["target"].to_numpy(np.float32)
id_pos = {k: i for i, k in enumerate(ids)}
n = len(y)
oof_lgbm = np.load(OOF_LGBM).astype(np.float32)
oof_tcn  = np.load(OOF_TCN).astype(np.float32)
print("rows:", n)

In [ ]:
# v13 OOF を parquet行順・v10空間(TVT-last_tvt)に整列
unet = np.load(OOF_UNET)                      # well -> pred_rel (ev順)
oof_unet = np.full(n, np.nan, np.float32)
miss = 0
for wid in unet.files:
    pred_rel = unet[wid].astype(np.float64)
    d = np.load(IMG_DIR / f"{wid}.npz")        # center,last_tvt,ev_index
    center = d["center"].astype(np.float64); last = float(d["last_tvt"]); ev = d["ev_index"]
    # 予測TVT = center + pred_rel → v10空間 = -last_tvt
    pred_v10 = (center + pred_rel - last).astype(np.float32)
    for k in range(len(ev)):
        pos = id_pos.get(f"{wid}_{int(ev[k])}")
        if pos is None: miss += 1; continue
        oof_unet[pos] = pred_v10[k]
print(f"v13整列 完了。未対応id={miss}, 欠損={np.isnan(oof_unet).sum()}")

In [ ]:
def rmse(p, m):
    return float(np.sqrt(np.mean((p[m]-y[m])**2)))

# 3つとも有効な行で評価
m = np.isfinite(oof_lgbm) & np.isfinite(oof_tcn) & np.isfinite(oof_unet) & np.isfinite(y)
print(f"評価行: {m.sum()} / {n}")
print(f"v10 LGBM : {rmse(oof_lgbm, m):.4f}")
print(f"v12 TCN  : {rmse(oof_tcn,  m):.4f}")
print(f"v13 UNet : {rmse(oof_unet, m):.4f}")
C = np.corrcoef(np.stack([oof_lgbm[m], oof_tcn[m], oof_unet[m]]))
print("OOF相関(lgbm,tcn,unet):\n", np.round(C, 3))

In [ ]:
# 重みグリッド探索(w10+w12+w13=1, step=0.05)
ya = y[m]; a = oof_lgbm[m]; b = oof_tcn[m]; c = oof_unet[m]
best = (1.0, 0.0, 0.0, rmse(oof_lgbm, m))
grid = np.arange(0, 1.0001, 0.05)
for w10 in grid:
    for w12 in grid:
        w13 = 1 - w10 - w12
        if w13 < -1e-9: continue
        pred = w10*a + w12*b + w13*c
        r = float(np.sqrt(np.mean((pred-ya)**2)))
        if r < best[3]: best = (w10, w12, w13, r)
w10, w12, w13, r = best
print(f"best weights  LGBM={w10:.2f}  TCN={w12:.2f}  UNet={w13:.2f}")
print(f"3-way blend overall RMSE = {r:.4f}")
print(f"  vs v10単独 {rmse(oof_lgbm,m):.4f} (Δ={r-rmse(oof_lgbm,m):+.4f})")
print(f"  vs v10×v12 ブレンド 7.3924(参考)")